In [3]:
import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv()

def get_connection():
    return mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        ssl_ca=os.getenv("DB_SSL_CA")
    )

# Prueba
conn = get_connection()
conn.close()

In [4]:
import pandas as pd
import google.genai as genai
from tabulate import tabulate
from IPython.display import display


In [5]:
from google import genai

client = genai.Client(api_key=os.getenv("LLM_API_KEY"))

In [ ]:
SYSTEM_PROMPT = """
Sos un experto en bases de datos MySQL. Tu única tarea es convertir preguntas 
en español a consultas SQL válidas para la base de datos 'biblioia'.

REGLAS:
- Respondé ÚNICAMENTE con la consulta SQL, sin explicaciones ni comentarios.
- No uses bloques de código ni backticks.
- Preferí usar las VISTAS disponibles cuando corresponda.
- Si la pregunta no se puede responder con el esquema dado, respondé: 
  SELECT 'No puedo responder esa pregunta con los datos disponibles';

=== ESQUEMA ===

GENERO (id_genero INT PK, nombre VARCHAR(60) UNIQUE NOT NULL, descripcion VARCHAR(255))
AUTOR (id_autor INT PK, nombre VARCHAR(80) NOT NULL, apellido VARCHAR(80) NOT NULL, nacionalidad VARCHAR(60))
LIBRO (isbn VARCHAR(20) PK, titulo VARCHAR(200) NOT NULL, anio_publicacion YEAR, stock_total SMALLINT, stock_disponible SMALLINT)
  -- stock_disponible <= stock_total siempre
LIBRO_AUTOR (isbn FK->LIBRO, id_autor FK->AUTOR) -- N:M
LIBRO_GENERO (isbn FK->LIBRO, id_genero FK->GENERO) -- N:M
SOCIO (id_socio INT PK, dni VARCHAR(15) UNIQUE, nombre VARCHAR(80), apellido VARCHAR(80), email VARCHAR(120) UNIQUE, fecha_alta DATE, estado VARCHAR(12))
  -- estado: 'ACTIVO', 'SUSPENDIDO', 'BAJA'
EJEMPLAR (id_ejemplar INT PK, isbn FK->LIBRO, nro_ejemplar SMALLINT, estado_fisico VARCHAR(12))
  -- estado_fisico: 'BUENO', 'DETERIORADO', 'BAJA'
PRESTAMO (id_prestamo INT PK, id_socio FK->SOCIO, id_ejemplar FK->EJEMPLAR, fecha_prestamo DATE, fecha_vencimiento DATE, fecha_devolucion DATE NULL, estado VARCHAR(12))
  -- estado: 'ACTIVO', 'DEVUELTO', 'VENCIDO'
SANCION (id_sancion INT PK, id_socio FK->SOCIO, tipo VARCHAR(20), fecha_inicio DATE, fecha_fin DATE, motivo VARCHAR(255))
  -- tipo: 'MORA', 'DAÑO', 'PERDIDA', 'OTRO'
  -- activa cuando fecha_fin >= CURRENT_DATE
AUDITORIA_PRESTAMOS (id_audit INT PK, id_prestamo INT, operacion VARCHAR(10), estado_nuevo VARCHAR(12), estado_viejo VARCHAR(12), usuario_bd VARCHAR(80), fecha_hora DATETIME)

=== VISTAS DISPONIBLES ===

v_prestamos_vencidos (id_prestamo, dni, socio, titulo, fecha_prestamo, fecha_vencimiento, dias_de_mora)
  -- préstamos con estado='VENCIDO' y sin devolución

v_prestamos_activos (id_prestamo, id_socio, dni, socio, email, isbn, titulo, nro_ejemplar, fecha_prestamo, fecha_vencimiento, dias_vencido)
  -- todos los préstamos con estado='ACTIVO'

v_libros_disponibles (isbn, titulo, stock_disponible, generos, autores)
  -- libros con stock_disponible > 0 y al menos un ejemplar en buen estado

v_historial_socios (id_socio, dni, socio, isbn, titulo, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado_prestamo)
  -- historial completo de préstamos por socio

v_libros_mas_prestados (isbn, titulo, autores, total_prestamos)
  -- ranking de libros por cantidad de préstamos

v_socios_sancionados (id_socio, dni, socio, estado, tipo, fecha_inicio, fecha_fin, motivo, dias_restantes)
  -- socios con sanciones activas hoy

v_autores_prolíficos (id_autor, autor, nacionalidad, cantidad_libros)
  -- autores con más de 1 libro en la biblioteca

v_lista_socios (lista_socios)
-- contine todas las filas de la tabla SOCIO

v_reporte_salud_biblioteca (total_inventario, ejemplares_circulacion, tasa_ocupacion, total_socios, porcentaje_socios_sancionados)
  -- contiene una única fila con las métricas generales y el estado de salud en tiempo real de la biblioteca

=== EJEMPLOS ===

Pregunta: ¿Cuáles son los 5 libros más prestados este año?
SQL: SELECT isbn, titulo, total_prestamos FROM v_libros_mas_prestados LIMIT 5;

Pregunta: ¿Qué socios tienen préstamos vencidos en este momento?
SQL: SELECT DISTINCT dni, socio FROM v_prestamos_vencidos;

Pregunta: ¿Qué libros de ciencia ficción están disponibles para prestar?
SQL: SELECT isbn, titulo, stock_disponible FROM v_libros_disponibles WHERE generos LIKE '%Ciencia Ficción%';
"""

In [7]:
def text_to_sql(pregunta: str) -> str:
    respuesta = client.models.generate_content(
       model="gemini-2.5-flash",
        contents=SYSTEM_PROMPT + f"\nPregunta: {pregunta}\nSQL:"
    )
    sql = respuesta.text.strip()
    # Por las dudas, limpiamos backticks que Gemini a veces agrega
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


In [8]:
def ejecutar_consulta(sql: str) -> pd.DataFrame:
    conn = get_connection()
    try:
        df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})
    # cirra la conexion 
    finally:
        conn.close()

In [9]:
def agente_responder(pregunta: str, mostrar_sql: bool = True):
    print(f"\n Pregunta: {pregunta}")
    print("-" * 60)
    
    sql = text_to_sql(pregunta)
    
    if mostrar_sql:
        print(f" SQL generado:\n{sql}")
        print("-" * 60)
    
    df = ejecutar_consulta(sql)
    
    if df.empty:
        print(" La consulta no devolvió resultados.")
    else:
        print(f" Resultado ({len(df)} filas):")
        display(df)
    
    return df

In [10]:
def obtener_perfil_socio(id_socio: int) -> dict:
    conn = get_connection()
    try:
        # Géneros que leyó
        df_generos = pd.read_sql(
            "SELECT genero FROM v_generos_por_socio WHERE id_socio = %s",
            conn, params=(id_socio,)
        )
        # Autores que leyó
        df_autores = pd.read_sql(
            "SELECT autor FROM v_autores_por_socio WHERE id_socio = %s",
            conn, params=(id_socio,)
        )
        return {
            "generos": df_generos['genero'].tolist(),
            "autores": df_autores['autor'].tolist()
        }
    finally:
        conn.close()

In [11]:
def obtener_libros_candidatos(id_socio: int) -> pd.DataFrame:
    conn = get_connection()
    try:
        df = pd.read_sql(
            "SELECT isbn, titulo, stock_disponible FROM v_libros_no_leidos_por_socio WHERE id_socio = %s",
            conn, params=(id_socio,)
        )
        return df
    finally:
        conn.close()

In [12]:
def recomendar_para(id_socio: int):
    print(f"\n Recomendaciones para el socio ID {id_socio}")
    print("-" * 60)
    
    perfil = obtener_perfil_socio(id_socio)
    
    if not perfil['generos']:
        print("Este socio no tiene historial de préstamos.")
        return
    
    print(f" Géneros favoritos: {', '.join(perfil['generos'])}")
    print(f" Autores leídos: {', '.join(perfil['autores'])}")
    
    df_candidatos = obtener_libros_candidatos(id_socio)
    
    if df_candidatos.empty:
        print(" No hay libros disponibles para recomendar.")
        return

    libros_str = df_candidatos[['titulo']].to_string(index=False)
    
    prompt = f"""
Sos un bibliotecario amigable . Un socio tiene estos gustos:
- Géneros favoritos: {', '.join(perfil['generos'])}
- Autores que ya leyó: {', '.join(perfil['autores'])}

Estos libros están disponibles y el socio aún no los leyó:
{libros_str}

Recomendá los 3 mejores. 
Para cada uno indicá el título y explicá en 2-3 oraciones por qué le va a gustar.
"""
    
    respuesta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    print("\n Recomendaciones:")
    print("=" * 60)
    print(respuesta.text)

In [14]:
recomendar_para(1)


 Recomendaciones para el socio ID 1
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_19840\896487870.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_generos = pd.read_sql(
C:\Users\miran\AppData\Local\Temp\ipykernel_19840\896487870.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_autores = pd.read_sql(


 Géneros favoritos: Ciencia Ficción
 Autores leídos: Jorge Luis Borges, Isaac Asimov


C:\Users\miran\AppData\Local\Temp\ipykernel_19840\1794459765.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(



 Recomendaciones:
¡Hola! ¡Qué gusto verte por aquí! Con esos gustos tan interesantes, me entusiasma ayudarte a encontrar tu próxima gran lectura. Has explorado las mentes brillantes de Borges y Asimov, así que buscamos ciencia ficción que estimule la mente y expanda el universo.

Aquí tienes mis 3 mejores recomendaciones, ¡creo que te van a encantar!

1.  **La mano izquierda de la oscuridad**
    Ursula K. Le Guin es una maestra, y este libro es una joya. Te sumergirás en un planeta donde los habitantes son andróginos y cambian de sexo, explorando temas profundos sobre género, sociedad e identidad. Es una ciencia ficción muy reflexiva, que combina la profundidad filosófica que aprecias en Borges con la construcción de mundos complejos al estilo Asimov, pero con una sensibilidad literaria única.

2.  **2001: Una odisea del espacio**
    Si disfrutas de la ciencia ficción de "grandes ideas" y la exploración cósmica de Asimov, Arthur C. Clarke es tu siguiente parada obligatoria. Este lib

In [15]:
def obtener_libros_comunidad(id_socio: int) -> pd.DataFrame:
    conn = get_connection()
    try:
        query = """
        SELECT DISTINCT l.titulo, g.nombre AS genero
        FROM PRESTAMO p
        JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
        JOIN LIBRO l ON e.isbn = l.isbn
        JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn
        JOIN GENERO g ON lg.id_genero = g.id_genero
        WHERE p.id_socio IN (
            -- Socios afines: leyeron los mismos géneros que el socio actual
            SELECT DISTINCT p2.id_socio 
            FROM PRESTAMO p2
            JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar
            JOIN LIBRO_GENERO lg2 ON e2.isbn = lg2.isbn
            WHERE lg2.id_genero IN (
                SELECT lg3.id_genero FROM PRESTAMO p3
                JOIN EJEMPLAR e3 ON p3.id_ejemplar = e3.id_ejemplar
                JOIN LIBRO_GENERO lg3 ON e3.isbn = lg3.isbn
                WHERE p3.id_socio = %s
            ) AND p2.id_socio != %s
        )
        AND l.isbn NOT IN (
            -- Excluir los libros que el socio actual YA leyó
            SELECT DISTINCT e4.isbn FROM PRESTAMO p4
            JOIN EJEMPLAR e4 ON p4.id_ejemplar = e4.id_ejemplar
            WHERE p4.id_socio = %s
        )
        AND l.stock_disponible > 0
        LIMIT 6;
        """
        df = pd.read_sql(query, conn, params=(id_socio, id_socio, id_socio))
        return df
    finally:
        conn.close()

In [ ]:
def recomendar_colaborativo(id_socio: int):
    print(f"\n  Recomendación Colaborativa para el Socio ID {id_socio}")
    print("-" * 65)
    
    # Buscamos los libros candidatos de la comunidad
    df_candidatos = obtener_libros_comunidad(id_socio)
    
    if df_candidatos.empty:
        print("No se encontraron suficientes coincidencias en la comunidad para este socio.")
        return

    # Formateamos los libros para el prompt
    libros_str = ""
    for _, fila in df_candidatos.iterrows():
        libros_str += f"- {fila['titulo']} (Género: {fila['genero']})\n"
    
    prompt_colaborativo = f"""
    Sos un bibliotecario amigable de 'BiblioIA'. Tu tarea es recomendar libros usando el método colaborativo.
    
    Otros lectores con gustos muy similares a este socio han estado leyendo y recomendando estos libros:
    {libros_str}
    
    Elegí los 3 mejores para sugerirle al socio actual (que todavía no los leyó).
    Para cada uno indicá el título y explicale en 2 oraciones por qué le va a gustar, mencionando de forma entusiasta que "otros lectores de la comunidad con sus mismos gustos" los recomendaron.
    """
    
    respuesta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt_colaborativo
    )
    
    print("\n Recomendaciones de la Comunidad:")
    print("=" * 65)
    print(respuesta.text)

In [18]:
recomendar_colaborativo(1)


 👥 BONUS: Recomendación Colaborativa para el Socio ID 1
-----------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_19840\2536482430.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params=(id_socio, id_socio, id_socio))



 Recomendaciones de la Comunidad:
¡Hola, socio! Soy tu amigable bibliotecario de BiblioIA, y tengo unas recomendaciones fantásticas para ti, ¡basadas en lo que otros lectores de la comunidad con tus mismos gustos han devorado y amado!

Aquí tienes 3 sugerencias que seguro te encantarán:

1.  **La mano izquierda de la oscuridad**
    ¡Este es un clásico imperdible! Otros lectores de la comunidad con tus mismos gustos han elogiado su profunda exploración de la sociedad, la identidad y la cultura. Te sumergirá en un fascinante mundo alienígena donde los conceptos de género se desdibujan, invitándote a reflexionar sobre la naturaleza humana.

2.  **Los desposeídos**
    ¡Una joya de la ciencia ficción social! Otros lectores de la comunidad con tus mismos gustos lo han recomendado efusivamente por su inteligencia y profundidad. Este libro te hará viajar entre dos planetas, explorando a fondo los ideales anarquistas y las complejidades de diferentes sistemas políticos y sociales.

3.  **Cit

In [21]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Creamos los componentes visuales
texto_pregunta = widgets.Textarea(
    value='',
    placeholder='Escribí tu pregunta acá...',
    description='Pregunta:',
    layout=widgets.Layout(width='70%', height='60px')
)

boton_buscar = widgets.Button(
    description='Preguntar al Agente',
    button_style='primary', # Le da un color azul estético
    icon='search'
)

# Contenedor para que los resultados no se acumulen infinitamente
area_salida = widgets.Output()

# 2. Definimos la función que se ejecuta al hacer clic
def al_hacer_clic(b):
    with area_salida:
        clear_output(wait=True) # Limpia el resultado anterior
        pregunta = texto_pregunta.value.strip()
        
        if not pregunta:
            print("❌ Por favor, ingresá una pregunta.")
            return
            
        # Llamamos a tu función principal existente
        agente_responder(pregunta, mostrar_sql=True)

# 3. Vinculamos el botón con la función
boton_buscar.on_click(al_hacer_clic)


# 4. Mostramos la interfaz en el notebook
print("🤖 ¡Bienvenido a la interfaz de BiblioIA! Escribí tu consulta en lenguaje natural:")
display(widgets.HBox([texto_pregunta, boton_buscar]))
display(area_salida)

🤖 ¡Bienvenido a la interfaz de BiblioIA! Escribí tu consulta en lenguaje natural:


Output()